In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install vector-quantize-pytorch sentence-transformers
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git
!pip install transformers

from google.colab import files
from google.colab import drive
import zipfile
import sys
import torch
import numpy as np
import pickle
import os
from datetime import datetime

#drive.mount('/content/drive')

!cp /content/drive/MyDrive/gen_rec_files_round_2.zip .

with zipfile.ZipFile("gen_rec_files_round_2.zip", 'r') as zip_ref:
    zip_ref.extractall("files")

sys.path.append("files")

from DataHandler import get_article_feature_string_list
from embedder import Embedder
from rqvae import RQVAE
from rqvae_train import train_rqvae_full
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device: {device}')

In [3]:
import os
os.chdir("files")

In [ ]:
feature_strings = get_article_feature_string_list()
# feature_strings is a list of lists, use list comprehension to make it a list of strings
feature_strings = [item[0] for item in feature_strings]
item_ids = [feature_string.split()[0] for feature_string in feature_strings]

print(f'Number of items: {len(feature_strings)}')

In [ ]:
if os.path.exists('SBERT_embeddings_fulldata.npy'):
    embeddings = np.load('SBERT_embeddings_fulldata.npy')
    print(f'The embeddings have the shape: {embeddings.shape}')
else:
    print("retrieving embeddings...")
    embedder = Embedder("bert-large")
    embeddings = embedder.encode(feature_strings) # embeddings has shape (n, d), where d = 384 for SBERT and d = 1024 for bert-large
    print(f'The embeddings have the shape: {embeddings.shape}')
    # print(embeddings[0])
    np.save('SBERT_embeddings_fulldata.npy', embeddings)

embeddings_numpy = embeddings.copy()   # this is the preserved NumPy version
np.save('bert-large_embeddings.npy', embeddings_numpy)

if isinstance(embeddings, np.ndarray):
    embeddings = torch.from_numpy(embeddings).float()
embeddings = embeddings.to(device)

In [ ]:
files.download("bert-large_embeddings.npy")

In [ ]:
input_dim = embeddings.shape[1]
latent_dim = 64
hidden_dim = 512
codebook_size = 512
num_quantizers = 5

os.makedirs('./models', exist_ok=True)

rqvae = RQVAE(input_dim=input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim, codebook_size=codebook_size, num_quantizers=num_quantizers)
rqvae = rqvae.to(device)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
trained_rqvae = train_rqvae_full(rqvae, embeddings, epochs=100, batch_size=256, save_path=f'./models/rqvae_{timestamp}', early_stopping_patience=15, min_delta=1e-4)

In [ ]:
semantic_ids = trained_rqvae.encode_to_semantic_ids(embeddings)
semantic_ids = semantic_ids.cpu().detach().numpy()

item_2_semantic = {}
semantic_2_item = {}
for item_id, semantic_id in zip(item_ids, semantic_ids):
    semantic_tuple = tuple(semantic_id.astype(int))
    offset = 0
    semantic_tuple = (*semantic_tuple, offset)

    while semantic_tuple in semantic_2_item:
        offset += 1
        semantic_tuple = (*semantic_tuple[:-1], offset)

    item_2_semantic[item_id] = semantic_tuple
    semantic_2_item[semantic_tuple] = item_id

print(f'Done creating hashmaps. Saving...')
with open(f'item_2_semantic_{timestamp}.pkl', 'wb') as f:
    pickle.dump(item_2_semantic, f)

with open(f'semantic_2_item_{timestamp}.pkl', 'wb') as f:
    pickle.dump(semantic_2_item, f)
print(f'Saved hashmaps to files.')

In [ ]:
from google.colab import files
import zipfile

zip_filename = f'rqvae_training_results_{timestamp}.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in os.listdir('./models/'):
        if timestamp in file:
            zipf.write(f'./models/{file}', f'models/{file}')

    zipf.write(f'item_2_semantic_{timestamp}.pkl', f'item_2_semantic_{timestamp}.pkl')
    zipf.write(f'semantic_2_item_{timestamp}.pkl', f'semantic_2_item_{timestamp}.pkl')

print(f"Created {zip_filename}")
files.download(zip_filename)

In [ ]:
from google.colab import drive
drive.flush_and_unmount()